In [20]:
# run this cell for sanity check 
import sys
print(sys.executable)

# It should print a path containing venv — that's your confirmation everything is wired up correctly. Let me know what you see!

c:\Users\pradheepa jaya shree\python.exe


In [21]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
import shap
import warnings
#-------------------------
# karthi made updation to imports here 
import random
#---------------------------
warnings.filterwarnings('ignore')

In [22]:
def load_all_patients(set_paths):
    dfs = []
    for folder in set_paths:
        files = os.listdir(folder)
        print(f"Loading {len(files)} files from {folder}...")
        for fname in files:
            if fname.endswith('.psv'):
                path = os.path.join(folder, fname)
                df = pd.read_csv(path, sep='|')
                df['patient_id'] = fname.replace('.psv', '')
                dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal rows: {len(combined)}")
    print(f"Total patients: {combined['patient_id'].nunique()}")
    return combined

# Point these to your actual folders
SET_A = r'C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA'
SET_B = r'C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setB'

data = load_all_patients([SET_A, SET_B])
data.head()

Loading 20336 files from C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA...
Loading 20000 files from C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setB...

Total rows: 1552210
Total patients: 40336


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001


In [23]:
label_counts = data['SepsisLabel'].value_counts()
print("SepsisLabel counts:")
print(label_counts)

neg = label_counts[0]
pos = label_counts[1]
scale_pos_weight = neg / pos

print(f"\nNon-sepsis rows: {neg}")
print(f"Sepsis rows:     {pos}")
print(f"Ratio:           {scale_pos_weight:.1f}x")
print(f"\nscale_pos_weight to use in XGBoost: {scale_pos_weight:.2f}")

SepsisLabel counts:
SepsisLabel
0    1524294
1      27916
Name: count, dtype: int64

Non-sepsis rows: 1524294
Sepsis rows:     27916
Ratio:           54.6x

scale_pos_weight to use in XGBoost: 54.60


In [24]:
def preprocess(df):
    df = df.copy()
    # #-------- old code was causing issue -----
    # # ── 1. Forward fill within each patient, then backward fill ──
    # df = df.groupby('patient_id', group_keys=False).apply(
    #     lambda x: x.ffill().bfill()
    # )



    # #-------karthi updated new code ---------------
    # # ── 1. Forward fill within each patient, then backward fill ──
    # Save patient_id first because pandas 2.x drops groupby keys after apply()
    patient_ids = df['patient_id'].copy()

    df = df.groupby('patient_id', group_keys=False).apply(
        lambda x: x.ffill().bfill()
    )

    # Restore patient_id if pandas dropped it (happens in pandas 2.0+)
    if 'patient_id' not in df.columns:
        df['patient_id'] = patient_ids
    # #---------------------------------------------
    
    # ── 2. Fill anything still missing with column median ──
    df = df.fillna(df.median(numeric_only=True))
    
    # ── 3. Safety net ──
    df = df.fillna(0)
    
    # ── 4. Rolling mean (last 6 hours) per patient ──
    vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp']
    for col in vitals:
        df[f'{col}_6h_mean'] = (
            df.groupby('patient_id')[col]
            .transform(lambda x: x.rolling(6, min_periods=1).mean())
        )
        df[f'{col}_3h_mean'] = (
            df.groupby('patient_id')[col]
            .transform(lambda x: x.rolling(3, min_periods=1).mean())
        )
    
    # ── 5. Delta features (change from previous hour) ──
    for col in vitals:
        df[f'{col}_delta'] = df.groupby('patient_id')[col].diff().fillna(0)
    
    # ── 6. Ratio feature (shock index) ──
    df['shock_index'] = df['HR'] / (df['SBP'].replace(0, np.nan).fillna(1))
    
    return df

print("Running preprocessing... (this takes 2-4 minutes)")
data = preprocess(data)
print("Done!")
print(f"Total features now: {len(data.columns)}")

Running preprocessing... (this takes 2-4 minutes)
Done!
Total features now: 61


In [25]:
FEATURES = [
    # Raw vitals
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp',
    # Key labs
    'WBC', 'Creatinine', 'Lactate', 'Glucose',
    'Potassium', 'HCO3', 'pH', 'Platelets',
    # Demographics
    'Age', 'Gender', 'HospAdmTime', 'ICULOS',
    # Engineered features
    'HR_6h_mean', 'Resp_6h_mean', 'Temp_6h_mean',
    'SBP_6h_mean', 'O2Sat_6h_mean',
    'HR_3h_mean', 'Resp_3h_mean',
    'HR_delta', 'Resp_delta', 'Temp_delta', 'SBP_delta',
    'shock_index'
]

X = data[FEATURES]
y = data['SepsisLabel']
groups = data['patient_id']

# Split by patient — NOT by row
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
X_test  = X.iloc[test_idx]
y_test  = y.iloc[test_idx]

print(f"Train rows: {len(X_train)}")
print(f"Test rows:  {len(X_test)}")
print(f"Sepsis in train: {y_train.sum()}")
print(f"Sepsis in test:  {y_test.sum()}")

Train rows: 1241213
Test rows:  310997
Sepsis in train: 22669
Sepsis in test:  5247


In [26]:
print(f"Training with scale_pos_weight = {scale_pos_weight:.2f}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1  # use all CPU cores
)

model.fit(X_train, y_train)
print("Training complete!")

Training with scale_pos_weight = 54.60
Training complete!


In [27]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"AUROC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nTarget: AUROC > 0.85")

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.87      0.93    305750
           1       0.07      0.61      0.13      5247

    accuracy                           0.86    310997
   macro avg       0.53      0.74      0.53    310997
weighted avg       0.98      0.86      0.91    310997

AUROC: 0.8271

Target: AUROC > 0.85


In [28]:
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/sepsis_xgb_model.pkl')
json.dump(FEATURES, open('models/feature_columns.json', 'w'))

print("Saved:")
print("  models/sepsis_xgb_model.pkl")
print("  models/feature_columns.json")

Saved:
  models/sepsis_xgb_model.pkl
  models/feature_columns.json


In [29]:
explainer = shap.TreeExplainer(model)

# Test on one row
sample = X_test.iloc[[0]]
shap_vals = explainer.shap_values(sample)

# Top 6 features for that row
shap_series = pd.Series(shap_vals[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("Top 6 contributing features for this prediction:")
print(shap_series.head(6))

Top 6 contributing features for this prediction:
Creatinine      0.609375
ICULOS         -0.418583
WBC            -0.208529
Potassium      -0.196945
MAP             0.186081
Temp_6h_mean   -0.181090
dtype: float32


In [30]:
# Cell 10 — Save population medians for API use
import json

medians = data[FEATURES].median().to_dict()
with open('models/population_medians.json', 'w') as f:
    json.dump(medians, f)

print("Saved models/population_medians.json")

Saved models/population_medians.json


In [31]:
import os, random

# Find a patient who actually got sepsis
sepsis_files = []
for folder in [SET_A, SET_B]:
    for fname in os.listdir(folder):
        if fname.endswith('.psv'):
            df_temp = pd.read_csv(os.path.join(folder, fname), sep='|')
            if df_temp['SepsisLabel'].sum() > 0:
                sepsis_files.append(os.path.join(folder, fname))

# Pick a random sepsis patient
test_file = random.choice(sepsis_files[:100])
print(f"Testing on: {test_file}")

# Load and preprocess
patient_df = pd.read_csv(test_file, sep='|')
patient_df['patient_id'] = 'test'
processed = preprocess(patient_df)

# Predict
X_patient = processed[FEATURES]
scores = model.predict_proba(X_patient)[:, 1]

print(f"\nHours in ICU: {len(scores)}")
print(f"Max risk score: {scores.max():.3f}")
print(f"Hour of peak risk: {scores.argmax()}")
print(f"Alert triggered (>0.65): {(scores > 0.65).any()}")
print(f"\nRisk scores by hour:")
for i, s in enumerate(scores):
    bar = '█' * int(s * 20)
    flag = ' ⚠️ ALERT' if s > 0.65 else ''
    print(f"  Hour {i+1:3d}: {s:.3f} {bar}{flag}")

Testing on: C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA\p000371.psv

Hours in ICU: 11
Max risk score: 0.070
Hour of peak risk: 10
Alert triggered (>0.65): False

Risk scores by hour:
  Hour   1: 0.010 
  Hour   2: 0.014 
  Hour   3: 0.013 
  Hour   4: 0.030 
  Hour   5: 0.036 
  Hour   6: 0.023 
  Hour   7: 0.024 
  Hour   8: 0.020 
  Hour   9: 0.058 █
  Hour  10: 0.044 
  Hour  11: 0.070 █


In [32]:
# Find a patient who never got sepsis
healthy_files = []
for folder in [SET_A, SET_B]:
    for fname in os.listdir(folder):
        if fname.endswith('.psv'):
            df_temp = pd.read_csv(os.path.join(folder, fname), sep='|')
            if df_temp['SepsisLabel'].sum() == 0:
                healthy_files.append(os.path.join(folder, fname))

test_file_healthy = random.choice(healthy_files[:100])
print(f"Testing on: {test_file_healthy}")

patient_df2 = pd.read_csv(test_file_healthy, sep='|')
patient_df2['patient_id'] = 'test'
processed2 = preprocess(patient_df2)
X_patient2 = processed2[FEATURES]
scores2 = model.predict_proba(X_patient2)[:, 1]

print(f"\nMax risk score: {scores2.max():.3f}")
print(f"Alert triggered (>0.65): {(scores2 > 0.65).any()}")
print("✅ Good: no false alarm" if not (scores2 > 0.65).any() else "⚠️ False alarm!")

Testing on: C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA\p000108.psv

Max risk score: 0.202
Alert triggered (>0.65): False
✅ Good: no false alarm


In [15]:
import os, random, joblib, json
import pandas as pd
import numpy as np

# ── point these to your actual folders ──────────────────────────────
# ── point these to your actual folders ──────────────────────────────
SET_A = r'C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA'
SET_B = r'C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setB'

# ← CHANGED: full absolute paths
MODEL_PATH    = r'C:\Users\pradheepa jaya shree\Desktop\Sepsis_detector\sepsis_detector\backend\models\sepsis_xgb_model.pkl'
FEATURES_PATH = r'C:\Users\pradheepa jaya shree\Desktop\Sepsis_detector\sepsis_detector\backend\models\feature_columns.json'
DEMO_FOLDER   = r'C:\Users\pradheepa jaya shree\Desktop\Sepsis_detector\sepsis_detector\backend\demo_data'
# ────────────────────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────

model    = joblib.load(MODEL_PATH)
FEATURES = json.load(open(FEATURES_PATH))
os.makedirs(DEMO_FOLDER, exist_ok=True)

# --- collect all sepsis patient file paths --------------------------
all_files = []
for folder in [SET_A, SET_B]:
    for fname in os.listdir(folder):
        if fname.endswith('.psv'):
            all_files.append(os.path.join(folder, fname))

sepsis_files = []
for path in all_files:
    df_tmp = pd.read_csv(path, sep='|')
    if 'SepsisLabel' in df_tmp.columns and df_tmp['SepsisLabel'].sum() > 0:
        sepsis_files.append(path)

print(f"Total sepsis patients found: {len(sepsis_files)}")

# --- score 50 random ones, pick the most dramatic -------------------
sample = random.sample(sepsis_files, min(50, len(sepsis_files)))

best_score  = -1
best_path   = None

for path in sample:
    df_tmp = pd.read_csv(path, sep='|')
    df_tmp['patient_id'] = 'test'
    processed  = preprocess(df_tmp)           # your existing preprocess fn
    available  = [f for f in FEATURES if f in processed.columns]
    X_tmp      = processed[available]
    scores     = model.predict_proba(X_tmp)[:, 1]
    
    # "dramatic" = low start, high finish, big rise
    if len(scores) >= 10:
        rise = scores[-5:].mean() - scores[:5].mean()
        if rise > best_score:
            best_score = rise
            best_path  = path

print(f"Best patient: {best_path}  (rise score: {best_score:.3f})")

# --- copy it to demo_data as csv ------------------------------------
df_sepsis = pd.read_csv(best_path, sep='|')
df_sepsis.to_csv(f'{DEMO_FOLDER}/sepsis_patient_DEMO.csv', index=False)
print("Saved: demo_data/sepsis_patient_DEMO.csv")

# --- pick a healthy patient (no sepsis label ever = 1) --------------
# --- pick a BETTER healthy patient (stays safely below 0.5 entire stay, short ICU) ---
print("\nSearching for ideal healthy patient...")

healthy_candidates = []
for p in all_files:
    df_tmp = pd.read_csv(p, sep='|')
    if df_tmp['SepsisLabel'].sum() == 0 and len(df_tmp) <= 30:
        healthy_candidates.append(p)

print(f"Found {len(healthy_candidates)} short-stay healthy candidates")

best_healthy_path = None
best_max_score = 999

sample_healthy = random.sample(healthy_candidates, min(100, len(healthy_candidates)))

for path in sample_healthy:
    df_tmp = pd.read_csv(path, sep='|')
    df_tmp['patient_id'] = 'test'
    processed = preprocess(df_tmp)
    available = [f for f in FEATURES if f in processed.columns]
    X_tmp = processed[available]
    scores = model.predict_proba(X_tmp)[:, 1]
    
    max_score = scores.max()
    if max_score < best_max_score:
        best_max_score = max_score
        best_healthy_path = path

print(f"Best healthy patient: {best_healthy_path}")
print(f"Max risk score across entire stay: {best_max_score:.3f}")

if best_max_score > 0.5:
    print("⚠️ Warning: couldn't find a patient below 0.5 — try running again")
else:
    print("✅ Good healthy patient found!")

df_healthy = pd.read_csv(best_healthy_path, sep='|')
df_healthy.to_csv(f'{DEMO_FOLDER}/healthy_patient_DEMO.csv', index=False)
print("Saved: demo_data/healthy_patient_DEMO.csv")

print("\nDone! Your demo_data folder now has exactly 2 files.")

Total sepsis patients found: 2932
Best patient: C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setB\p109148.psv  (rise score: 0.846)
Saved: demo_data/sepsis_patient_DEMO.csv

Searching for ideal healthy patient...
Found 12847 short-stay healthy candidates
Best healthy patient: C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA\p001709.psv
Max risk score across entire stay: 0.000
✅ Good healthy patient found!
Saved: demo_data/healthy_patient_DEMO.csv

Done! Your demo_data folder now has exactly 2 files.


In [16]:
import joblib, json, pandas as pd
import numpy as np

MODEL_PATH    = r'C:\Users\pradheepa jaya shree\Desktop\Sepsis_detector\sepsis_detector\backend\models\sepsis_xgb_model.pkl'
FEATURES_PATH = r'C:\Users\pradheepa jaya shree\Desktop\Sepsis_detector\sepsis_detector\backend\models\feature_columns.json'
SEPSIS_DEMO   = r'C:\Users\pradheepa jaya shree\Desktop\Sepsis_detector\sepsis_detector\backend\demo_data\sepsis_patient_DEMO.csv'
HEALTHY_DEMO  = r'C:\Users\pradheepa jaya shree\Desktop\Sepsis_detector\sepsis_detector\backend\demo_data\healthy_patient_DEMO.csv'

model    = joblib.load(MODEL_PATH)
FEATURES = json.load(open(FEATURES_PATH))

def test_patient(filepath, label):
    df = pd.read_csv(filepath)
    df['patient_id'] = 'test'
    processed = preprocess(df)
    available = [f for f in FEATURES if f in processed.columns]
    X = processed[available]
    scores = model.predict_proba(X)[:, 1]
    
    print(f"\n{'='*50}")
    print(f"Patient: {label}")
    print(f"Hours in ICU: {len(scores)}")
    print(f"Max risk score: {scores.max():.3f}")
    print(f"Min risk score: {scores.min():.3f}")
    print(f"Alert triggered (>0.65): {(scores > 0.65).any()}")
    print(f"\nRisk by hour:")
    for i, s in enumerate(scores):
        bar = '█' * int(s * 30)
        flag = ' ⚠️' if s > 0.65 else ''
        print(f"  Hour {i+1:3d}: {s:.3f} |{bar}{flag}")
    
    # Verdict
    print(f"\n{'✅ PASS' if check_pass(scores, label) else '❌ FAIL'}")

def check_pass(scores, label):
    if label == 'SEPSIS':
        return (scores > 0.65).any()       # must trigger alert
    else:
        return scores.max() < 0.5          # must never trigger

test_patient(SEPSIS_DEMO,  'SEPSIS')
test_patient(HEALTHY_DEMO, 'HEALTHY')


Patient: SEPSIS
Hours in ICU: 96
Max risk score: 0.981
Min risk score: 0.056
Alert triggered (>0.65): True

Risk by hour:
  Hour   1: 0.056 |█
  Hour   2: 0.105 |███
  Hour   3: 0.065 |█
  Hour   4: 0.190 |█████
  Hour   5: 0.183 |█████
  Hour   6: 0.183 |█████
  Hour   7: 0.266 |███████
  Hour   8: 0.171 |█████
  Hour   9: 0.206 |██████
  Hour  10: 0.224 |██████
  Hour  11: 0.119 |███
  Hour  12: 0.130 |███
  Hour  13: 0.122 |███
  Hour  14: 0.228 |██████
  Hour  15: 0.282 |████████
  Hour  16: 0.263 |███████
  Hour  17: 0.298 |████████
  Hour  18: 0.359 |██████████
  Hour  19: 0.338 |██████████
  Hour  20: 0.466 |█████████████
  Hour  21: 0.341 |██████████
  Hour  22: 0.487 |██████████████
  Hour  23: 0.529 |███████████████
  Hour  24: 0.283 |████████
  Hour  25: 0.330 |█████████
  Hour  26: 0.242 |███████
  Hour  27: 0.143 |████
  Hour  28: 0.163 |████
  Hour  29: 0.139 |████
  Hour  30: 0.105 |███
  Hour  31: 0.100 |██
  Hour  32: 0.145 |████
  Hour  33: 0.195 |█████
  Hour  34: 0

In [17]:
test_patient(HEALTHY_DEMO, 'HEALTHY')


Patient: HEALTHY
Hours in ICU: 8
Max risk score: 0.000
Min risk score: 0.000
Alert triggered (>0.65): False

Risk by hour:
  Hour   1: 0.000 |
  Hour   2: 0.000 |
  Hour   3: 0.000 |
  Hour   4: 0.000 |
  Hour   5: 0.000 |
  Hour   6: 0.000 |
  Hour   7: 0.000 |
  Hour   8: 0.000 |

✅ PASS


In [18]:
healthy_candidates_long = []
for p in all_files:
    df_tmp = pd.read_csv(p, sep='|')
    if df_tmp['SepsisLabel'].sum() == 0 and 20 <= len(df_tmp) <= 40:
        healthy_candidates_long.append(p)

print(f"Found {len(healthy_candidates_long)} candidates")

best_healthy_path = None
best_score_diff = 999

sample_h = random.sample(healthy_candidates_long, min(150, len(healthy_candidates_long)))

for path in sample_h:
    df_tmp = pd.read_csv(path, sep='|')
    df_tmp['patient_id'] = 'test'
    processed = preprocess(df_tmp)
    available = [f for f in FEATURES if f in processed.columns]
    X_tmp = processed[available]
    scores = model.predict_proba(X_tmp)[:, 1]
    
    max_s = scores.max()
    # Want max between 0.1-0.4 (shows variation but no false alarm)
    if 0.05 < max_s < 0.40:
        best_healthy_path = path
        best_max_score = max_s
        break

print(f"Patient: {best_healthy_path}")
print(f"Max score: {best_max_score:.3f}")

df_healthy = pd.read_csv(best_healthy_path, sep='|')
df_healthy.to_csv(HEALTHY_DEMO, index=False)
print("✅ Saved new healthy patient")

Found 16540 candidates
Patient: C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA\p018578.psv
Max score: 0.077
✅ Saved new healthy patient


In [19]:
test_patient(HEALTHY_DEMO, 'HEALTHY')


Patient: HEALTHY
Hours in ICU: 28
Max risk score: 0.077
Min risk score: 0.034
Alert triggered (>0.65): False

Risk by hour:
  Hour   1: 0.051 |█
  Hour   2: 0.051 |█
  Hour   3: 0.064 |█
  Hour   4: 0.061 |█
  Hour   5: 0.066 |█
  Hour   6: 0.077 |██
  Hour   7: 0.034 |█
  Hour   8: 0.048 |█
  Hour   9: 0.059 |█
  Hour  10: 0.043 |█
  Hour  11: 0.057 |█
  Hour  12: 0.059 |█
  Hour  13: 0.063 |█
  Hour  14: 0.059 |█
  Hour  15: 0.050 |█
  Hour  16: 0.052 |█
  Hour  17: 0.051 |█
  Hour  18: 0.052 |█
  Hour  19: 0.055 |█
  Hour  20: 0.072 |██
  Hour  21: 0.069 |██
  Hour  22: 0.053 |█
  Hour  23: 0.054 |█
  Hour  24: 0.054 |█
  Hour  25: 0.051 |█
  Hour  26: 0.039 |█
  Hour  27: 0.053 |█
  Hour  28: 0.046 |█

✅ PASS
